In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.metrics import precision_score, recall_score, f1_score, mean_squared_error, mean_absolute_error
import math

In [2]:
import zipfile

# Define the path to the zip file and the output directory
zip_path = "southeast.csv.zip"
output_dir = "./"  # Extract to the current directory

# Open and extract the zip file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)

print("Unzipping complete!")


Unzipping complete!


In [3]:




# ---- Select only required columns ----
usecols = ['Data', 'Hora', 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
           'UMIDADE RELATIVA DO AR, HORARIA (%)',
           'VENTO, VELOCIDADE HORARIA (m/s)']

dtype_optimization = {col: 'float32' for col in usecols[2:]}


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [14]:
# ---- Load data in chunks ----
chunk_size = 100_000
data_list = []

for chunk in pd.read_csv('southeast.csv', usecols=usecols, dtype=dtype_optimization, chunksize=chunk_size):
    chunk.replace(-9999, np.nan, inplace=True)
    chunk.fillna(method='ffill', inplace=True)
    chunk.drop(columns=['Data', 'Hora'], inplace=True)
    data_list.append(chunk.values)

data = np.vstack(data_list)
del data_list

print(f"Data shape: {data.shape}")  # Debug memory usage


<ipython-input-14-11b116d22712>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  chunk.fillna(method='ffill', inplace=True)
<ipython-input-14-11b116d22712>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  chunk.fillna(method='ffill', inplace=True)
<ipython-input-14-11b116d22712>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  chunk.fillna(method='ffill', inplace=True)
<ipython-input-14-11b116d22712>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  chunk.fillna(method='ffill', inplace=True)
<ipython-input-14-11b116d22712>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use o

Data shape: (15345216, 3)


In [17]:
data = data[~np.isnan(data).any(axis=1)]


In [20]:

# ---- Create Lazy Sequences ----
SEQ_LEN = 24
PRED_HORIZON = 6

def create_sequences_lazy(data, seq_len, horizon, step=1):
    for i in range(0, len(data) - seq_len - horizon + 1, step):
        yield data[i:i+seq_len], data[i+seq_len:i+seq_len+horizon]

n = len(data)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_generator = create_sequences_lazy(data[:train_end], SEQ_LEN, PRED_HORIZON)
val_generator = create_sequences_lazy(data[train_end:val_end], SEQ_LEN, PRED_HORIZON)
test_generator = create_sequences_lazy(data[val_end:], SEQ_LEN, PRED_HORIZON)



In [21]:

# ---- Custom Iterable Dataset ----
class LazyDataset(Dataset):
    def __init__(self, data_gen):
        self.data_gen = list(data_gen)

    def __len__(self):
        return len(self.data_gen)

    def __getitem__(self, idx):
        x, y = self.data_gen[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_dataset = LazyDataset(train_generator)
val_dataset = LazyDataset(val_generator)
test_dataset = LazyDataset(test_generator)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("DataLoader is ready!")

DataLoader is ready!


In [26]:
# VAE for Anomaly Detection (Optimized)
train_size = int(0.7 * len(data))
val_size = int(0.85 * len(data))

# Avoid using `.values` on NumPy arrays
X_vae_train = data[:train_size]
X_vae_val = data[train_size:val_size]
X_vae_test = data[val_size:]

# Custom Iterable Dataset for Large Data Handling
class VAEIterableDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32).to(device)

# Use iterable datasets to avoid memory overuse
vae_train_dataset = VAEIterableDataset(X_vae_train)
vae_val_dataset = VAEIterableDataset(X_vae_val)
vae_test_dataset = VAEIterableDataset(X_vae_test)

vae_train_loader = DataLoader(vae_train_dataset, batch_size=64, shuffle=True)
vae_val_loader = DataLoader(vae_val_dataset, batch_size=64, shuffle=False)
vae_test_loader = DataLoader(vae_test_dataset, batch_size=64, shuffle=False)

print("VAE DataLoader is ready!")


VAE DataLoader is ready!


In [22]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc31 = nn.Linear(32, latent_dim)
        self.fc32 = nn.Linear(32, latent_dim)
        self.fc4 = nn.Linear(latent_dim, 32)
        self.fc5 = nn.Linear(32, 64)
        self.fc6 = nn.Linear(64, input_dim)
    def encode(self, x):
        h1 = torch.relu(self.fc1(x))
        h2 = torch.relu(self.fc2(h1))
        return self.fc31(h2), self.fc32(h2)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        h4 = torch.relu(self.fc4(z))
        h5 = torch.relu(self.fc5(h4))
        return self.fc6(h5)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar, z

In [23]:
def vae_loss(recon, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

In [24]:
def train_vae(model, train_loader, val_loader, num_epochs=50, patience=5):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    best_state = model.state_dict()
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:  # Remove (batch,) to avoid unpacking issue
            batch = batch.to(device)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(batch)
            loss = vae_loss(recon, batch, mu, logvar)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:  # Remove (batch,)
                batch = batch.to(device)
                recon, mu, logvar, _ = model(batch)
                loss = vae_loss(recon, batch, mu, logvar)
                val_loss += loss.item()
        val_loss /= len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)
    return train_losses, val_losses


In [27]:
input_dim = data.shape[1]  # Correct feature count
latent_dim = 8

vae_model = VAE(input_dim, latent_dim).to(device)

# Train the VAE
vae_train_losses, vae_val_losses = train_vae(vae_model, vae_train_loader, vae_val_loader)

# Plot loss curve
plt.figure()
plt.plot(vae_train_losses, label='Train Loss')
plt.plot(vae_val_losses, label='Val Loss')
plt.legend()
plt.title('VAE Training Loss')
plt.savefig('vae_loss.png')

print("VAE training complete. Loss plot saved!")


KeyboardInterrupt: 

In [17]:
def compute_reconstruction_errors(model, loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch in loader:
            recon, _, _, _ = model(batch)
            batch_errors = torch.mean((recon - batch) ** 2, dim=1)
            errors.extend(batch_errors.cpu().numpy())
    return np.array(errors)
train_errors = compute_reconstruction_errors(vae_model, vae_train_loader)
threshold = np.percentile(train_errors, 95)
test_errors = compute_reconstruction_errors(vae_model, vae_test_loader)
anomalies = test_errors > threshold
train_temp = X_vae_train[:, 0].numpy()
temp_mean = np.mean(train_temp)
temp_std = np.std(train_temp)

ValueError: too many values to unpack (expected 1)

In [ ]:
def label_anomaly(x):
    return (x[0] > temp_mean + 2 * temp_std) or (x[0] < temp_mean - 2 * temp_std)
gt_labels = np.array([label_anomaly(x) for x in X_vae_test.numpy()])
prec = precision_score(gt_labels, anomalies)
rec = recall_score(gt_labels, anomalies)
f1 = f1_score(gt_labels, anomalies)
print("VAE Anomaly Detection - Precision:", prec, "Recall:", rec, "F1:", f1)
vae_model.eval()
latents = []
with torch.no_grad():
    for (batch,) in vae_test_loader:
        _, _, _, z = vae_model(batch)
        latents.append(z)
latents = torch.cat(latents, dim=0).cpu().numpy()
pca = PCA(n_components=2)
latents_2d = pca.fit_transform(latents)
plt.figure(); plt.scatter(latents_2d[:, 0], latents_2d[:, 1], c=anomalies, cmap='coolwarm', s=2); plt.title('Latent Space'); plt.savefig('latent_space.png')
z_sample = torch.randn(100, latent_dim)
synthetic_data = vae_model.decode(z_sample).detach().cpu().numpy()

In [ ]:
# GRU Forecasting
class GRUForecast(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(GRUForecast, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, target=None, teacher_forcing_ratio=0.5):
        batch_size = x.size(0)
        outputs = []
        hidden = None
        input_t = x[:, -1, :].unsqueeze(1)
        for t in range(PRED_HORIZON):
            out, hidden = self.gru(input_t, hidden)
            out_step = self.fc(out.squeeze(1))
            outputs.append(out_step.unsqueeze(1))
            if target is not None and np.random.rand() < teacher_forcing_ratio:
                input_t = target[:, t, :].unsqueeze(1)
            else:
                input_t = out_step.unsqueeze(1)
        return torch.cat(outputs, dim=1)

In [ ]:
class LSTMForecast(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTMForecast, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, target=None, teacher_forcing_ratio=0.5):
        batch_size = x.size(0)
        outputs = []
        hidden = None
        input_t = x[:, -1, :].unsqueeze(1)
        for t in range(PRED_HORIZON):
            out, hidden = self.lstm(input_t, hidden)
            out_step = self.fc(out.squeeze(1))
            outputs.append(out_step.unsqueeze(1))
            if target is not None and np.random.rand() < teacher_forcing_ratio:
                input_t = target[:, t, :].unsqueeze(1)
            else:
                input_t = out_step.unsqueeze(1)
        return torch.cat(outputs, dim=1)

In [ ]:

class RNNForecast(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(RNNForecast, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, target=None, teacher_forcing_ratio=0.5):
        batch_size = x.size(0)
        outputs = []
        hidden = None
        input_t = x[:, -1, :].unsqueeze(1)
        for t in range(PRED_HORIZON):
            out, hidden = self.rnn(input_t, hidden)
            out_step = self.fc(out.squeeze(1))
            outputs.append(out_step.unsqueeze(1))
            if target is not None and np.random.rand() < teacher_forcing_ratio:
                input_t = target[:, t, :].unsqueeze(1)
            else:
                input_t = out_step.unsqueeze(1)
        return torch.cat(outputs, dim=1)

In [ ]:
def train_forecast(model, train_loader, val_loader, num_epochs=30):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []
    teacher_forcing_ratio = 1.0
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch, target=Y_batch, teacher_forcing_ratio=teacher_forcing_ratio)
            loss = criterion(output, Y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, Y_batch in val_loader:
                X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
                output = model(X_batch, teacher_forcing_ratio=0)
                loss = criterion(output, Y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        teacher_forcing_ratio = max(0.0, teacher_forcing_ratio - 0.05)
        scheduler.step()
    return train_losses, val_losses

In [ ]:
gru_model = GRUForecast(input_dim, 64, input_dim).to(device)
gru_train_losses, gru_val_losses = train_forecast(gru_model, train_loader, val_loader)
plt.figure(); plt.plot(gru_train_losses, label='Train Loss'); plt.plot(gru_val_losses, label='Val Loss'); plt.legend(); plt.savefig('gru_loss.png')


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            output = model(X_batch, teacher_forcing_ratio=0)
            preds.append(output.cpu().numpy())
            actuals.append(Y_batch.cpu().numpy())
    preds = np.concatenate(preds, axis=0)
    actuals = np.concatenate(actuals, axis=0)
    mse_val = mean_squared_error(actuals.flatten(), preds.flatten())
    return mse_val

In [ ]:
gru_mse = evaluate_model(gru_model, test_loader).to(device)
preds = []
actuals = []
with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        output = gru_model(X_batch, teacher_forcing_ratio=0)
        preds.append(output.cpu().numpy())
        actuals.append(Y_batch.cpu().numpy())
preds = np.concatenate(preds, axis=0)
actuals = np.concatenate(actuals, axis=0)
mse = mean_squared_error(actuals.flatten(), preds.flatten())
mae = mean_absolute_error(actuals.flatten(), preds.flatten())
rmse = math.sqrt(mse)
print("GRU Forecasting - MSE:", mse, "MAE:", mae, "RMSE:", rmse)
plt.figure(); plt.plot(actuals.flatten(), label='Actual'); plt.plot(preds.flatten(), label='Predicted'); plt.legend(); plt.savefig('forecast_vs_actual.png')


In [ ]:
lstm_model = LSTMForecast(input_dim, 64, input_dim).to(device)
1

In [ ]:
rnn_model = RNNForecast(input_dim, 64, input_dim).to(device)


In [ ]:
train_forecast(lstm_model, train_loader, val_loader)


In [ ]:
train_forecast(rnn_model, train_loader, val_loader)

In [ ]:
gru_mse_ablation = evaluate_model(gru_model, test_loader)


In [ ]:
lstm_mse_ablation = evaluate_model(lstm_model, test_loader)

In [ ]:
rnn_mse_ablation = evaluate_model(rnn_model, test_loader)


In [ ]:
print("Ablation - GRU MSE:", gru_mse_ablation, "LSTM MSE:", lstm_mse_ablation, "RNN MSE:", rnn_mse_ablation)


In [ ]:

# Model Integration
X_test_flat = X_test.reshape(-1, input_dim)
X_test_tensor = torch.tensor(X_test_flat, dtype=torch.float32).to(device)
recon, _, _, _ = vae_model(X_test_tensor)
reconstruction_errors = torch.mean((recon - X_test_tensor) ** 2, dim=1).cpu().numpy()
threshold_seq = np.percentile(train_errors, 95)
seq_errors = np.mean(reconstruction_errors.reshape(-1, SEQ_LEN), axis=1)
normal_indices = seq_errors <= threshold_seq
X_test_normal = X_test[normal_indices]
Y_test_normal = Y_test[normal_indices]
if len(X_test_normal) > 0:
    normal_dataset = TimeSeriesDataset(X_test_normal, Y_test_normal)
    normal_loader = DataLoader(normal_dataset, batch_size=64, shuffle=False)
    normal_mse = evaluate_model(gru_model, normal_loader)
    print("GRU on normal data MSE:", normal_mse)
